In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import plotly.express as px
import torch 
import torch.nn as nn
import sys
from pathlib import Path
from tqdm import tqdm

In [7]:
import re

In [8]:
training_data = []

for path in sorted(Path("../datasets").glob("*.txt")):
    text = path.read_text(encoding="utf-8")
    conversations = re.split(r"\n\s*\n", text.strip())
    for conversation in conversations:
        tokens = ["<START>"]
        for line in conversation.splitlines():
            if line.strip():
                words = re.findall(r"\w+|[^\w\s]", line)
                tokens.extend(words)
                tokens.append("<END>")
        if len(tokens) > 1:
            training_data.extend(tokens)

print(training_data[:300])


['<START>', 'You', 'enjoyed', 'yourself', 'in', 'America', '<END>', 'Eh', '<END>', 'did', 'you', '<END>', 'Oh', 'I', 'covered', 'a', 'nice', 'trip', 'yes', '<END>', 'Oh', 'very', 'good', '<END>', 'saw', 'Mary', 'and', 'Andrew', 'and', '<END>', 'Yes', 'you', 'did', '<END>', 'in', 'fact', 'the', 'whole', 'family', 'was', 'together', 'for', 'Mary', "'", 's', 'wedding', '<END>', 'Oh', 'very', 'nice', 'very', 'nice', 'yes', '<END>', 'It', "'", 's', 'horrible', '<END>', 'It', 'is', 'horrible', 'isn', "'", 't', '<END>', 'Have', 'you', 'been', 'busy', '<END>', 'Yes', '<END>', 'Yes', 'oh', '<END>', 'Jim', "'", 's', 'been', 'for', 'a', 'this', 'afternoon', 'at', 'the', 'H', 'art', 'and', 'Straw', 'Club', '<END>', 'oh', 'not', 'very', 'well', 'we', 'er', 'm', 'we', 'stopped', 'going', 'after', 'Christmas', 'because', 'we', 'had', 'bad', 'chest', 's', 'both', 'of', 'us', '<END>', 'Oh', '<END>', 'both', 'cold', 'and', 'it', "'", 's', 'hard', 'going', 'that', 'three', 'hours', 'in', 'the', 'morning'

In [9]:
vocab = sorted(set(training_data))
vocab.append("<UNK>")
vocab_pd = pd.DataFrame({
    "index": range(len(vocab)),
    "vocab": vocab
})



In [11]:
vocab_pd.to_csv("vocab.csv", index=False)

In [ ]:
import random

In [ ]:
import Trans_model

In [ ]:
import VanillaAgent

In [ ]:
def getTrainingData(batchsize = 5000, inputsize = 40):
    input = []
    target = []
    for i in range(batchsize):
        random_num = random.randint(0,len(training_data)-(inputsize+1))
        vocabList = training_data[random_num : random_num+inputsize+1]
        temp = []
        for j in vocabList:
            temp.append(VanillaAgent.vocab_to_token[j])
        input.append(temp[0:-1])
        target.append(temp[1:])

    return torch.tensor(input), torch.tensor(target)


In [16]:
input, target = getTrainingData()
input

tensor([[50291, 45534,     0,  ..., 72235, 60364, 73101],
        [49889, 62383, 73837,  ..., 65652, 73066, 35626],
        [71790, 73281,    13,  ...,  8768, 78433, 46603],
        ...,
        [  976, 29741, 78025,  ..., 65128, 73909,    13],
        [54141, 35203, 59264,  ..., 33667, 73316, 60364],
        [59569,    13,  4981,  ..., 78433, 72144, 51780]])

In [19]:
trans_model = Trans_model()
optimizer = torch.optim.AdamW(
    trans_model.parameters(),
    lr= 0.001,
    weight_decay= 0.0001
)

def trainingloop(iteration=256, batchsize=256, inputsize=64):
    loss_graph = []
    device = next(trans_model.parameters()).device

    trans_model.train()
    progress = tqdm(range(iteration))

    for step in progress:
        inputs, targets = getTrainingData(
            batchsize=batchsize,
            inputsize=inputsize
        )

        inputs = inputs.to(device=device, dtype=torch.long)
        targets = targets.to(device=device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)

        # [batch, sequence_length, vocab_size]
        logits = trans_model(inputs)

        # Compare every position with its next-token target.
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        loss_value = loss.item()
        loss_graph.append(loss_value)
        progress.set_postfix(loss=f"{loss_value:.4f}")

    return loss_graph

        

In [20]:
loss_graph = trainingloop()

100%|██████████| 256/256 [08:06<00:00,  1.90s/it, loss=5.2435]


In [22]:
index = np.arange(0,len(loss_graph),1)
plt.figure(figsize=(12,8))
plot = px.scatter(x = index, y= loss_graph)
plot.show()

<Figure size 1200x800 with 0 Axes>

In [23]:
torch.save(
    trans_model.state_dict(),
    "transformer_model.pth"
)